
# Variational Autoencoders (VAE)
## Deep Theoretical + Mathematical + Intuitive Walkthrough

This notebook is built to:

• Provide full theoretical understanding  
• Show the math clearly  
• Include executable PyTorch examples  
• Clarify earlier confusions  
• Highlight where intuition was correct  
• Highlight where corrections were needed  

We move step‑by‑step from fundamentals to gradients.

---



# 1️⃣ Deterministic Autoencoder vs VAE

### Deterministic Autoencoder

z = f(x)

Each input maps to one exact latent point.

Problem:
• No uncertainty  
• No generative structure  
• Latent space may fragment  

---

### Variational Autoencoder

Instead of:

z = f(x)

We define:

q_φ(z|x) = N(μ_φ(x), σ²_φ(x))

Meaning:

We do NOT map x to a single coordinate.

We map x to a probability distribution over latent coordinates.

✅ Earlier intuition that we compress into latent space — correct.  
❗ Correction: We compress into a *distribution*, not a single point.



# 2️⃣ What Does Probability Distribution Mean Here?

It does NOT mean:

• Probability of pixels  
• Probability of labels  
• μ values summing to 1  

Correction from earlier confusion:

μ and σ are NOT probability vectors.

They are parameters of a Gaussian probability density.

We model uncertainty over latent vector z.

For each input x:

μ(x) → center of latent region  
σ(x) → spread (uncertainty)  

Probability of WHAT?

Probability of different possible latent representations z.



# 3️⃣ Reparameterization Trick

Original (problematic):

z ~ N(μ, σ²)

Gradient cannot flow through sampling.

Rewritten:

ε ~ N(0,1)  
z = μ + σ * ε

Key insight:

Randomness no longer depends on parameters.

Before:
z = RANDOM(μ, σ)

After:
z = μ + σ * RANDOM(0,1)

This isolates randomness.

✅ Your understanding here was exactly correct.


In [ ]:

import torch

torch.manual_seed(0)

mu = torch.tensor([1.0], requires_grad=True)
log_var = torch.tensor([0.0], requires_grad=True)
sigma = torch.exp(0.5 * log_var)

epsilon = torch.randn(1)
z = mu + sigma * epsilon

loss = z**2  # simple scalar loss

loss.backward()

mu.grad, log_var.grad



# 4️⃣ KL Divergence (Regularization)

KL(q || p)

Where:

q = N(μ, σ²)  
p = N(0,1)

Closed form:

KL = 0.5 * Σ (μ² + σ² − log(σ²) − 1)

What it does:

• Pushes μ toward 0  
• Pushes σ toward 1  
• Prevents latent explosion  
• Forces global structure  

Correction:

We do NOT introduce prior because backprop fails.

We introduce prior to structure latent space.


In [ ]:

mu = torch.tensor([2.0])
sigma = torch.tensor([0.5])

kl = 0.5 * torch.sum(mu**2 + sigma**2 - torch.log(sigma**2) - 1)
kl



# 5️⃣ Full VAE Loss

Loss = Reconstruction Loss + KL Divergence

Reconstruction:
Forces similar inputs to cluster.

KL:
Forces global latent structure.

Epsilon:
Ensures smooth decoding and enables gradients.

Earlier intuition refinement:

Clustering emerges due to reconstruction pressure, not because Gaussian automatically clusters.



# 6️⃣ The Theoretical Gradient Step

We want:

∇_φ E_{z ~ q_φ}[L(z)]

Reparameterization gives:

∇_φ E_{ε ~ N(0,1)}[L(μ + σ ε)]

Because ε does NOT depend on φ:

= E_ε [ ∇_φ L(μ + σ ε) ]

This is valid because randomness is now independent of parameters.

Key idea:

We transformed stochastic sampling into deterministic transformation of fixed noise.

That is the core mathematical reason VAE training works.



# 7️⃣ Visualizing Latent Distribution

We show how μ and σ define a region, not a single point.


In [ ]:

import matplotlib.pyplot as plt

mu = torch.tensor([0.0, 0.0])
sigma = torch.tensor([1.0, 0.5])

samples = []
for _ in range(1000):
    eps = torch.randn(2)
    z = mu + sigma * eps
    samples.append(z)

samples = torch.stack(samples)

plt.figure()
plt.scatter(samples[:,0], samples[:,1], alpha=0.3)
plt.title("Latent Gaussian Cloud")
plt.show()



# 8️⃣ Final Intuition Summary

You were correct that:

• We compress input into latent structure  
• Clustering happens  
• KL acts like pinball stabilizer  
• Reparameterization isolates randomness  

Corrections clarified:

• Probability is over latent coordinates, not pixels  
• μ and σ are not classical statistics  
• Prior is for structure, not gradient fix  
• ε is random each forward pass, not constant  

Final big picture:

Encoder defines Gaussian cloud  
Sampling uses ε  
Decoder reconstructs  
KL shapes geometry  

Together → structured, smooth, generative latent space.
